# Banco de prueba (testbench) — MotorController + GRBL

Notebook para probar las funcionalidades básicas del controlador de motores usando **GRBL 1.1** sobre Arduino UNO + CNC Shield.

La idea es validar, en forma incremental:

- conexión serie
- estado de GRBL
- inicialización
- `Set Home` / `To Home`
- movimientos relativos y absolutos
- `jog` manual estilo `$J=`
- límites lógicos
- barridos de grilla

**Precaución:** comenzar con pasos chicos y velocidades bajas. Verificar que los motores no estén mecánicamente forzados antes de ejecutar movimientos.


## 1. Importaciones y configuración inicial

Cambiar `PORT` según el sistema:

- Linux / Raspberry Pi: `/dev/ttyUSB0` o `/dev/ttyACM0`
- Windows: `COM3`, `COM4`, `COM5`, etc.

La velocidad típica de GRBL es `115200` baudios.

In [25]:
# Import bibliotecas
import serial
import time

# Puerto serie: cambiar según corresponda
PORT = "/dev/ttyUSB0"   # Linux / Raspberry Pi
# PORT = "COM5"         # Windows
BAUDRATE = 115200
TIMEOUT = 2


## 2. Clase `MotorController`

Implementación actual de la clase. Esta versión incluye:

- movimientos relativos (`move_relative`)
- movimientos absolutos (`move_absolute`)
- jog manual (`jog` / `jog_cancel`)
- home lógico (`set_home` / `go_home`)
- límites lógicos
- barrido de grilla (`scan_grid`)

In [26]:

from motorcontroller import MotorController

## 3. Abrir conexión serie

Esta celda abre el puerto serie y crea el controlador. Si el puerto no existe o está ocupado, revisar que UGS u otro programa no lo tenga abierto.

In [27]:
ser = serial.Serial(PORT, BAUDRATE, timeout=TIMEOUT)
time.sleep(2.0)  # GRBL suele resetear al abrir puerto

# Inicializa controlador con mapeo de ejes: lógico x->GRBL X, lógico y->GRBL Z, lógico z no se usa.
controller = MotorController(
    ser,
    axis_map={
        "x": ("X",-1),  # eje lógico x invertido -> eje GRBL X
        "y": ("Z", 1),  # eje lógico y sin invertir -> eje GRBL Z
        "z": (None, 1)  # eje lógico z no se usa
    }
)

print("Arduino conectado en", ser.port)
print("Asingacion posiciones lógicas -> físicas:", controller.axis_map)
print("Motor Controller listo. Estado inicial:", controller.position)

Arduino conectado en /dev/ttyUSB0
Asingacion posiciones lógicas -> físicas: {'x': ('X', -1), 'y': ('Z', 1), 'z': (None, 1)}
Motor Controller listo. Estado inicial: {'x': 0.0, 'y': 0.0, 'z': 0.0}


## 4. Test de estado GRBL

Consulta el estado con `?`. Debería devolver algo tipo `Idle`, `Run`, `Alarm`, etc.

In [28]:
controller.wake_up()
print(controller.status())

<Idle|MPos:0.000,0.000,0.000|FS:0,0>


## 5. Inicialización básica

Ejecuta `$X`, `G21` y define una velocidad inicial. `$X` desbloquea GRBL si está en estado `Alarm`.

In [29]:
controller.initialize(feed=50.0)
print(controller.status())

<Idle|MPos:0.000,0.000,0.000|FS:0,0>


## 6. Configurar límites lógicos

Los límites evitan movimientos fuera de la región segura. Están activos por defecto.

Ajustar estos valores al rango seguro real del conjunto motor–acople antibacklash–tornillo.

In [31]:
controller.set_limits(
    x_min=-5.0, x_max=5.0,
    y_min=-5.0, y_max=5.0,
    z_min=-5.0, z_max=5.0,
)
controller.enable_limits()
print(controller.limits)
print("Límites activos:", controller.limits_enabled)

{'x_min': -5.0, 'x_max': 5.0, 'y_min': -5.0, 'y_max': 5.0, 'z_min': -5.0, 'z_max': 5.0}
Límites activos: True


## 7. Set Home

`set_home()` define la posición actual como `(0,0,0)` usando `G92 X0 Y0 Z0`.

Usarlo cuando el sistema esté en la posición que se quiere tomar como referencia inicial.

In [32]:
controller.set_home()
print(controller.position)

{'x': 0.0, 'y': 0.0, 'z': 0.0}


## 8. Test de movimiento relativo

Movimiento determinístico con `G1` en modo incremental. Es el modo recomendado para automatización y barridos.

Este ejemplo mueve el motor X asociado al eje lógico x un paso y vuelve. Luego probamos el motor Z asociado logicamente al eje y.

In [21]:
STEP = 1.0
FEED = 30.0

# Motor en modo relativo, mueve 1mm a la derecha, espera 0.5s, vuelve al origen.

# Movimiento eje x:X
controller.move_relative(dx=STEP, feed=FEED)
time.sleep(0.5)
controller.move_relative(dx=-STEP, feed=FEED)

# Movimiento eje y:Z
controller.move_relative(dy=STEP, feed=FEED)
time.sleep(0.5)
controller.move_relative(dy=-STEP, feed=FEED)

print(controller.position)

{'x': 0.0, 'y': 0.0, 'z': 0.0}


## 9. Test de ejes X, Y y Z

Prueba un pequeño movimiento positivo y negativo en cada eje. Usar valores chicos.

In [31]:
STEP = 1.0
FEED = 50.0

for axis in ["x", "y", "z"]:
    kwargs = {f"d{axis}": STEP, "feed": FEED}
    controller.move_relative(**kwargs)
    time.sleep(0.3)
    kwargs = {f"d{axis}": -STEP, "feed": FEED}
    controller.move_relative(**kwargs)
    time.sleep(0.3)

print(controller.position)

{'x': 0.0, 'y': 0.0, 'z': 0.0}


## 10. Test de movimiento absoluto y To Home

`move_absolute()` usa `G90`. `go_home()` vuelve al home lógico definido.

In [32]:
controller.move_absolute(x=3.0, y=0.5, z=3.0, feed=200.0)
print("Posición:", controller.position)

time.sleep(1.0)
print("Moving to home...")
for n in range(10):
    print(f"{10-n}..." , end=" ")
    time.sleep(0.5)
print("")
controller.go_home(feed=30.0)
print("Posición luego de go_home:", controller.position)

Posición: {'x': 3.0, 'y': 0.5, 'z': 3.0}
Moving to home...
10... 9... 8... 7... 6... 5... 4... 3... 2... 1... 
Posición luego de go_home: {'x': 0.0, 'y': 0.0, 'z': 0.0}


## 11. Test de `jog` manual

Usa `$J=G91...`. Es útil para control manual tipo joystick.

Para rutinas automáticas, conviene usar `move_relative()`.

In [33]:
controller.jog(dx=1.0, feed=20.0)
time.sleep(0.5)
controller.jog(dx=-1.0, feed=20.0)

print(controller.position)

Enviando comando de jogging: $J=G91 X-1.0000 F20.00
Enviando comando de jogging: $J=G91 X1.0000 F20.00
{'x': 0.0, 'y': 0.0, 'z': 0.0}


## 12. Test de cancelación de jog

`jog_cancel()` envía el comando realtime `0x85`. Es útil si después se implementa un joystick continuo.

In [34]:
# Ejemplo conservador: se lanza un jog pequeño y luego se cancela.
controller.jog(dx=1.0, feed=20.0)
time.sleep(2)
print("Jog canelling")
controller.jog_cancel()
print(controller.status())

Enviando comando de jogging: $J=G91 X-1.0000 F20.00
Jog canelling
<Jog|MPos:-0.656,0.000,0.000|FS:20,0>


In [35]:
# Volver al home lógico al finalizar pruebas
controller.go_home(feed=100.0)

'ok'

## 13. Test de límites lógicos

Este test intenta mover fuera del rango permitido. Debería generar una excepción `ValueError` y no mover el motor.

In [36]:
try:
    controller.move_absolute(x=999.0, y=0.0, z=0.0, feed=30.0)
except ValueError as exc:
    print("Límite detectado correctamente:")
    print(exc)

Límite detectado correctamente:
ALERTA: movimiento fuera de límites operativos. X=999.0000 mm fuera de rango [-5.0000, 5.0000]


## 14. Visualizar puntos de una grilla sin mover

Esto permite verificar el patrón antes de ejecutarlo físicamente.

In [41]:
points = controller._generate_grid_points(rows=8, cols=8, 
                                          step_x=0.5, step_y=0.5,
                                          drift_xy=0.02, drift_yx=-0.01,
                                          pattern="zigzag", centered=True)
for i, p in enumerate(points):
    print(i, p)

0 (-1.785, -1.732)
1 (-1.285, -1.738)
2 (-0.785, -1.742)
3 (-0.285, -1.748)
4 (0.215, -1.752)
5 (0.715, -1.758)
6 (1.215, -1.762)
7 (1.715, -1.768)
8 (1.725, -1.268)
9 (1.225, -1.262)
10 (0.725, -1.258)
11 (0.225, -1.252)
12 (-0.275, -1.248)
13 (-0.775, -1.242)
14 (-1.275, -1.238)
15 (-1.775, -1.232)
16 (-1.765, -0.733)
17 (-1.265, -0.738)
18 (-0.765, -0.743)
19 (-0.265, -0.748)
20 (0.235, -0.752)
21 (0.735, -0.757)
22 (1.235, -0.762)
23 (1.735, -0.767)
24 (1.745, -0.268)
25 (1.245, -0.263)
26 (0.745, -0.258)
27 (0.245, -0.253)
28 (-0.255, -0.247)
29 (-0.755, -0.242)
30 (-1.255, -0.237)
31 (-1.755, -0.232)
32 (-1.745, 0.268)
33 (-1.245, 0.263)
34 (-0.745, 0.258)
35 (-0.245, 0.253)
36 (0.255, 0.247)
37 (0.755, 0.242)
38 (1.255, 0.237)
39 (1.755, 0.232)
40 (1.765, 0.733)
41 (1.265, 0.738)
42 (0.765, 0.743)
43 (0.265, 0.748)
44 (-0.235, 0.752)
45 (-0.735, 0.757)
46 (-1.235, 0.762)
47 (-1.735, 0.767)
48 (-1.725, 1.268)
49 (-1.225, 1.262)
50 (-0.725, 1.258)
51 (-0.225, 1.252)
52 (0.275, 1.2

## 15. Barrido de grilla con delay

Recorre una grilla chica centrada alrededor del home. Usar `return_home=True` para volver al origen al finalizar.

In [43]:
controller.set_home()

controller.scan_grid(
    rows=3,
    cols=3,
    step_x=1.0,
    step_y=1.0,
    drift_xy=0.5,
    #drift_yx=-0.2,
    feed=100.0,
    pattern="zigzag",
    centered=False,
    wait_mode="delay",
    delay_s=0.5,
    return_home=True,
)

print(controller.position)

{'x': 0.0, 'y': 0.0, 'z': 0.0}


## 16. Barrido con callback de medición simulada

El callback se ejecuta en cada punto. Acá se simula una medición; luego se puede reemplazar por adquisición real del LIDAR.

In [44]:
measurements = []

def fake_measurement(i, x, y):
    print(f"Midiendo en punto {i}...")
    time.sleep(3.0)  # Simula tiempo de medición
    value = 1.0  # reemplazar por medición real
    measurements.append({"index": i, "x": x, "y": y, "value": value})
    print(f"Medición {i}: X={x:.4f}, Y={y:.4f}, value={value}")
    return {"ok": True}

controller.set_home()
controller.scan_grid(
    rows=3,
    cols=3,
    step_x=1,
    step_y=0.5,
    feed=50.0,
    pattern="raster",
    centered=True,
    wait_mode="callback",
    on_point=fake_measurement,
    return_home=True,
)

measurements

Midiendo en punto 0...
Medición 0: X=-1.0000, Y=-0.5000, value=1.0
Midiendo en punto 1...
Medición 1: X=0.0000, Y=-0.5000, value=1.0
Midiendo en punto 2...
Medición 2: X=1.0000, Y=-0.5000, value=1.0
Midiendo en punto 3...
Medición 3: X=-1.0000, Y=0.0000, value=1.0
Midiendo en punto 4...
Medición 4: X=0.0000, Y=0.0000, value=1.0
Midiendo en punto 5...
Medición 5: X=1.0000, Y=0.0000, value=1.0
Midiendo en punto 6...
Medición 6: X=-1.0000, Y=0.5000, value=1.0
Midiendo en punto 7...
Medición 7: X=0.0000, Y=0.5000, value=1.0
Midiendo en punto 8...
Medición 8: X=1.0000, Y=0.5000, value=1.0


[{'index': 0, 'x': -1.0, 'y': -0.5, 'value': 1.0},
 {'index': 1, 'x': 0.0, 'y': -0.5, 'value': 1.0},
 {'index': 2, 'x': 1.0, 'y': -0.5, 'value': 1.0},
 {'index': 3, 'x': -1.0, 'y': 0.0, 'value': 1.0},
 {'index': 4, 'x': 0.0, 'y': 0.0, 'value': 1.0},
 {'index': 5, 'x': 1.0, 'y': 0.0, 'value': 1.0},
 {'index': 6, 'x': -1.0, 'y': 0.5, 'value': 1.0},
 {'index': 7, 'x': 0.0, 'y': 0.5, 'value': 1.0},
 {'index': 8, 'x': 1.0, 'y': 0.5, 'value': 1.0}]

## 17. Barrido con fallo simulado

Este test simula que un punto falla en la tercera medición y pide acción manual. Para pruebas automáticas puede usarse `on_fail='skip'` o `on_fail='abort'`.

In [45]:
def fake_measurement_with_failure(i, x, y):
    print(f"Midiendo en punto {i}...")
    if i == 3:
        print(f"Fallo simulado en punto {i}!")
        return {"ok": False, "action": "wait_user", "reason": "fallo simulado"}
    return {"ok": True}

#controller.set_home()
controller.scan_grid(
    rows=3,
    cols=3,
    step_x=0.5,
    step_y=0.5,
    feed=20.0,
    pattern="zigzag",
    centered=True,
    wait_mode="callback",
    on_point=fake_measurement_with_failure,
    on_fail="skip",
    return_home=True,
)

print(controller.position)

Midiendo en punto 0...
Midiendo en punto 1...
Midiendo en punto 2...
Midiendo en punto 3...
Fallo simulado en punto 3!
Midiendo en punto 4...
Midiendo en punto 5...
Midiendo en punto 6...
Midiendo en punto 7...
Midiendo en punto 8...
{'x': 0.0, 'y': 0.0, 'z': 0.0}


## 18. Revisar archivos de estado e historial

Permite verificar qué quedó guardado en `motor_state.json` y las últimas líneas del historial.

In [ ]:
print("Estado actual:")
print(Path("motor_state.json").read_text(encoding="utf-8"))

print("\nÚltimos eventos:")
history = Path("motor_history.jsonl")
if history.exists():
    lines = history.read_text(encoding="utf-8").splitlines()
    for line in lines[-10:]:
        print(line)

Estado actual:
{
  "home": {
    "x": 0.0,
    "y": 0.0,
    "z": 0.0
  },
  "position": {
    "x": 0.0,
    "y": 0.0,
    "z": 0.0
  },
  "limits": {
    "x_min": -5.0,
    "x_max": 5.0,
    "y_min": -5.0,
    "y_max": 5.0,
    "z_min": -5.0,
    "z_max": 5.0
  },
  "limits_enabled": true,
  "updated_at": "2026-05-21T18:19:46Z"
}

Últimos eventos:
{"ts": "2026-05-21T18:18:45Z", "event": "move_absolute", "x": 2.25, "y": 3.75, "z": 0.0, "feed": 50.0}
{"ts": "2026-05-21T18:18:45Z", "event": "scan_grid_point", "index": 252, "x": 2.25, "y": 3.75, "z": 0.0}
{"ts": "2026-05-21T18:18:59Z", "event": "move_absolute", "x": 2.75, "y": 3.75, "z": 0.0, "feed": 50.0}
{"ts": "2026-05-21T18:18:59Z", "event": "scan_grid_point", "index": 253, "x": 2.75, "y": 3.75, "z": 0.0}
{"ts": "2026-05-21T18:19:13Z", "event": "move_absolute", "x": 3.25, "y": 3.75, "z": 0.0, "feed": 50.0}
{"ts": "2026-05-21T18:19:13Z", "event": "scan_grid_point", "index": 254, "x": 3.25, "y": 3.75, "z": 0.0}
{"ts": "2026-05-21T18:19:

## 19. Test: scan_grid_calibrated()

In [21]:
# ============================================================
# Test: scan_grid_calibrated()
# ============================================================
# Este test verifica que el wrapper calcule automáticamente:
# - step_x
# - step_y
# - drift_xy
# - drift_yx
#
# a partir de las cuatro esquinas calibradas, y luego ejecute
# scan_grid() usando esos parámetros.
#
# Recomendación:
# Primero ejecutar con wait_mode="user" o con una grilla chica.
# ============================================================

# Puntos medidos de calibración respecto del centro lógico (0,0)
#top_left = (8.3, 10.5)
#top_right = (-7.1, 10.5)
#bottom_left = (8.0, -10.9)
#bottom_right = (-7.6, -11.2)

top_left = (-8.0, 8.0)
top_right = (8.0, 8.0)
bottom_left = (-8.0, -8.0)
bottom_right = (8.0, -8.0)


rows = 5
cols = 5

# ------------------------------------------------------------
# 1. Calcular manualmente los parámetros para inspección
# ------------------------------------------------------------

width_top = top_right[0] - top_left[0]
width_bottom = bottom_right[0] - bottom_left[0]

height_left = bottom_left[1] - top_left[1]
height_right = bottom_right[1] - top_right[1]

avg_width = (width_top + width_bottom) / 2.0
avg_height = (height_left + height_right) / 2.0

step_x = abs(avg_width) / (cols - 1)
step_y = abs(avg_height) / (rows - 1)

drift_xy_left = (bottom_left[0] - top_left[0]) / avg_height
drift_xy_right = (bottom_right[0] - top_right[0]) / avg_height
drift_xy = (drift_xy_left + drift_xy_right) / 2.0

drift_yx_top = (top_right[1] - top_left[1]) / avg_width
drift_yx_bottom = (bottom_right[1] - bottom_left[1]) / avg_width
drift_yx = (drift_yx_top + drift_yx_bottom) / 2.0

print("Parámetros calculados:")
print(f"width_top     = {width_top:.3f}")
print(f"width_bottom  = {width_bottom:.3f}")
print(f"height_left   = {height_left:.3f}")
print(f"height_right  = {height_right:.3f}")
print(f"avg_width     = {avg_width:.3f}")
print(f"avg_height    = {avg_height:.3f}")
print(f"step_x        = {step_x:.3f}")
print(f"step_y        = {step_y:.3f}")
print(f"drift_xy      = {drift_xy:.6f}")
print(f"drift_yx      = {drift_yx:.6f}")


Parámetros calculados:
width_top     = 16.000
width_bottom  = 16.000
height_left   = -16.000
height_right  = -16.000
avg_width     = 16.000
avg_height    = -16.000
step_x        = 4.000
step_y        = 4.000
drift_xy      = -0.000000
drift_yx      = 0.000000


In [22]:
# ------------------------------------------------------------
# 2. Generar puntos usando scan_grid normal, sin mover motores
# ------------------------------------------------------------
# Esto permite inspeccionar qué puntos se recorrerían.
# Debe usar los parámetros calculados arriba.

points = controller._generate_grid_points(
    rows=rows,
    cols=cols,
    step_x=step_x,
    step_y=step_y,
    drift_xy=drift_xy,
    drift_yx=drift_yx,
    pattern="zigzag",
    centered=True
)

print("\nPuntos generados:")
for i, p in enumerate(points[:]):
    print(i, p)


Puntos generados:
0 (-8.0, -8.0)
1 (-4.0, -8.0)
2 (0.0, -8.0)
3 (4.0, -8.0)
4 (8.0, -8.0)
5 (8.0, -4.0)
6 (4.0, -4.0)
7 (0.0, -4.0)
8 (-4.0, -4.0)
9 (-8.0, -4.0)
10 (-8.0, 0.0)
11 (-4.0, 0.0)
12 (0.0, 0.0)
13 (4.0, 0.0)
14 (8.0, 0.0)
15 (8.0, 4.0)
16 (4.0, 4.0)
17 (0.0, 4.0)
18 (-4.0, 4.0)
19 (-8.0, 4.0)
20 (-8.0, 8.0)
21 (-4.0, 8.0)
22 (0.0, 8.0)
23 (4.0, 8.0)
24 (8.0, 8.0)


In [23]:
# ------------------------------------------------------------
# 3. Ejecutar barrido calibrado con callback de prueba
# ------------------------------------------------------------
# Este callback NO mide nada real; solo imprime la posición.
# Para prueba segura, se puede cambiar wait_mode="user".
# ------------------------------------------------------------

def test_callback(i, x, y):
    print(f"Punto {i:02d}: x={x:.3f}, y={y:.3f}")
    return {"ok": True}

# Asegurarse de tener un home lógico definido antes de empezar
controller.set_home()

controller.disable_limits()

controller.scan_grid_calibrated(
    rows=rows,
    cols=cols,
    top_left=top_left,
    top_right=top_right,
    bottom_left=bottom_left,
    bottom_right=bottom_right,
    feed=80.0,
    pattern="raster",
    centered=True,
    reverse=False,
    wait_mode="user",   # opciones: "none", "delay", "user", "callback"
    delay_s=0.0,
    on_point=test_callback,
    on_fail="abort",
    return_home=True,
)

Punto 00: x=-8.000, y=-8.000
Punto 01: x=-4.000, y=-8.000
Punto 02: x=0.000, y=-8.000
Punto 03: x=4.000, y=-8.000
Punto 04: x=8.000, y=-8.000
Punto 05: x=-8.000, y=-4.000
Punto 06: x=-4.000, y=-4.000
Punto 07: x=0.000, y=-4.000
Punto 08: x=4.000, y=-4.000
Punto 09: x=8.000, y=-4.000
Punto 10: x=-8.000, y=0.000
Punto 11: x=-4.000, y=0.000
Punto 12: x=0.000, y=0.000
Punto 13: x=4.000, y=0.000
Punto 14: x=8.000, y=0.000
Punto 15: x=-8.000, y=4.000
Punto 16: x=-4.000, y=4.000
Punto 17: x=0.000, y=4.000
Punto 18: x=4.000, y=4.000
Punto 19: x=8.000, y=4.000
Punto 20: x=-8.000, y=8.000
Punto 21: x=-4.000, y=8.000
Punto 22: x=0.000, y=8.000
Punto 23: x=4.000, y=8.000
Punto 24: x=8.000, y=8.000


## 20. Cerrar puerto serie

Cerrar el puerto serie al finalizar.

In [62]:
ser.close()
print("Puerto cerrado")

Puerto cerrado
